# Macrophage heterogeneity and spatial interactions in PDAC

This working copy presents the Python workflow developed independently by Davide Basile during a Cancer Research UK internship. PDAC refers to pancreatic ductal adenocarcinoma.

The original `MacrophagesAnalysisValidation.working.ipynb` is the unchanged record with saved outputs. This copy has no outputs and has not been executed after refactoring. See `README.md` for setup and `docs/REVIEW.md` for known bugs and scientific limitations.

1. Load and preprocess macrophage expression data.
2. Compare treated and untreated macrophages with the original Wilcoxon workflow.
3. Score the treatment-associated gene signature and select cells within each condition.
4. Calculate exponential spatial proximity scores and compare sample summaries.
5. Infer candidate ligand–receptor interactions with LIANA and quantify them with SCOTIA.

**Execution is opt-in.** Keep `RUN_ANALYSIS = False` to inspect this notebook safely. Full execution can be expensive. `RUN_LR_ANALYSIS` separately gates the LIANA/SCOTIA loop. No methodological fixes have been applied: in particular, mean tests retain the original median-based test selection for review.



### Part 1: Data Loading, DGE, and Subtype Definition<br>
---


#### 1.1: Initial Setup and Data Loading

In [ ]:
from pathlib import Path

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Launch Jupyter from the repository root. No analysis runs on package import.
RUN_ANALYSIS = False
RUN_LR_ANALYSIS = False
DATA_PATH = Path("raw_meta_data_final.h5ad")
OUTPUT_ROOT = Path("LigandReceptor_Analysis_Output")

from pdac_macrophages.preprocessing import (
    replace_nan_with_zero,
    sample_expression_zscores,
)


path to AnnData

In [ ]:
if not RUN_ANALYSIS:
    raise RuntimeError(
        "Analysis is disabled. Review README.md and docs/REVIEW.md before "
        "setting RUN_ANALYSIS = True."
    )
adata = sc.read(DATA_PATH)



Inspect data

In [ ]:
adata.obs


#### 1.2: Subset and Preprocess Macrophage Data

Subset to all macrophages 

In [ ]:
is_mac = (adata.obs["annotation_majortypes"] == "Macrophage")
mac = adata[is_mac].copy()

Verify what 'annotation_subtypes' holds 

In [ ]:
print("\nUnique macrophage subtypes found:")
print(mac.obs["annotation_subtypes"].unique().tolist())

Grab global X/Y coordinates for each macrophage

In [ ]:
x = mac.obs["CenterX_global_px"].astype(float)
y = mac.obs["CenterY_global_px"].astype(float)

Build a small Pandas DataFrame to compute stats per Macro cluster

In [ ]:
df_mac = pd.DataFrame({
    "subtype": mac.obs["annotation_subtypes"].values,
    "x": x.values,
    "y": y.values
})

Investigate macrophage subtypes by counting cells and computing median coordinate for each Macro_# cluster

In [ ]:
cluster_stats = (
    df_mac
    .groupby("subtype")
    .agg(
        n_cells=("subtype", "count"),
        median_x=("x", "median"),
        median_y=("y", "median")
    )
    .loc[["Macro_#1", "Macro_#2", "Macro_#3", "Macro_#4", "Macro_#5", "Macro_#6"]]
)

In [ ]:
print("\nMacrophage cluster statistics:")
print(cluster_stats)

In [ ]:
# Matrix preparation is provided by pdac_macrophages.preprocessing.


Make copy of macrophage data

In [ ]:
mac2 = mac.copy()

Replace NaNs with zero while keeping sparse matrices sparse. This preserves the original imputation rule.


In [ ]:
mac2.X = replace_nan_with_zero(mac2.X)


Define clean macrophage data without NaNs

In [ ]:
mac_clean =  mac2.copy()

Perform single-cell normalisation

In [ ]:
sc.pp.normalize_total(mac_clean, target_sum=1e4)
sc.pp.log1p(mac_clean)


#### 1.3: Differential Gene Expression (DGE) between Treated and Untreated Macrophages


Annotate cells by treatment condition

In [ ]:
id_list = mac.obs["sample_id"].cat.categories 
untreated_ids = [id for id in id_list if id.startswith("U") ] 
treated_ids = [id for id in id_list if id.startswith("T")] 
mac_clean.obs["treatment"] = mac.obs["sample_id"].isin(treated_ids).map({True:"Treated", False:"Untreated"}) 

Verify cell counts per condition

In [ ]:
mac_count = mac_clean.obs["treatment"].value_counts()
print(mac_count)
ratio = mac_count['Treated']/mac_count['Untreated']
print(f"Ratio treated to untreated macs is {ratio}")


Run Wilcoxon rank-sum test to find differentially expressed genes

In [ ]:
sc.tl.rank_genes_groups(
    mac_clean,
    groupby="treatment",
    groups=["Treated"],
    reference="rest",
    method="wilcoxon",
    pts=True
)

Extract and format the results into a DataFrame

In [ ]:
df = sc.get.rank_genes_groups_df(mac_clean, group="Treated")
df = df.rename(columns={
    "names": "gene",
    "logfoldchanges": "log2FC",
    "pvals_adj": "p_adj"
})


#### 1.4: Prepare Gene Lists for Visualization and Scoring


Get top 50 most differentiated genes based on absolute fold change

In [ ]:
N = 50
df["absFC"] = df["log2FC"].abs()
top_genes = (
    df
    .query("p_adj<=0.05 and absFC>=1.0")
    .sort_values("absFC", ascending=False)
    .head(N)["gene"]
    .tolist()
)
top_genes = [g for g in top_genes if g in mac_clean.var_names]

Compute per-sample mean expression and gene-wise z-scores (sample standard deviation).


In [ ]:
expr_z = sample_expression_zscores(
    mac_clean[:, top_genes].X, mac_clean.obs["sample_id"], top_genes
)



Get separate lists of significantly up- and down-regulated genes

In [ ]:
up_genes =(
    df
    .query("p_adj<=0.05 and log2FC>=1.0")
    .sort_values("log2FC", ascending=False)
    ["gene"]
    .tolist()
)
down_genes = (
    df
    .query("p_adj <= 0.05 and log2FC <= -1.0")
    .sort_values("log2FC", ascending=True)
    ["gene"]
    .tolist()
)
up_genes = [g for g in up_genes if g in mac_clean.var_names]
down_genes = [g for g in down_genes if g in mac_clean.var_names]

Apply the same sample-level z-score calculation to upregulated and downregulated gene sets separately.


In [ ]:
expr_z_up = sample_expression_zscores(
    mac_clean[:, up_genes].X, mac_clean.obs["sample_id"], up_genes
)
expr_z_down = sample_expression_zscores(
    mac_clean[:, down_genes].X, mac_clean.obs["sample_id"], down_genes
)



#### 1.5: Define and Apply Plotting Function for DGE Heatmaps

In [ ]:
# Plotting helpers below retain the original visualization settings.


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_DE(adata, expression_matrix_dataframe):
    # build treatment color strip keyed to each sample 
    lut = {"Untreated": "red", "Treated": "blue"}
    # `mapping` is a dict sample_id → "Untreated" or "Treated"
    mapping = (
        adata.obs[["sample_id","treatment"]] # collect sample ids and treatment columns 
        .astype({"sample_id": str})  #make sure sample id is a string type 
        .drop_duplicates("sample_id") #drop duplicates
        .set_index("sample_id")["treatment"]#set sample_id as index and bìchoo treatment column 
    )
    col_colors = [lut[mapping[s]] for s in expression_matrix_dataframe.columns]

    #Split sample lists by treatment, look at the sample columnn of the transpodes matrix data. 
    cols_unt = [s for s in expression_matrix_dataframe.columns if mapping[s]=="Untreated"]
    cols_tr  = [s for s in expression_matrix_dataframe.columns if mapping[s]=="Treated"]

    #Compute linkage & leaf order within each block
    link_unt = linkage(expression_matrix_dataframe[cols_unt].T, method="ward", metric="euclidean")
    order_unt = [cols_unt[i] for i in leaves_list(link_unt)] 
    link_tr  = linkage(expression_matrix_dataframe[cols_tr].T, method="ward", metric="euclidean")
    order_tr  = [cols_tr[i] for i in leaves_list(link_tr)]

    #New column order: all Untreated (in their cluster order), then Treated
    new_cols = order_unt + order_tr

    #Re‐index your data and rebuild the colour bar
    data_ordered   = expression_matrix_dataframe[new_cols] 
    col_colors = [lut[mapping[s]] for s in new_cols]

    #Draw the clustermap with columns fixed
    cg = sns.clustermap(
        data_ordered,
        cmap="viridis",
        vmin=-1,         # floor the colors at –1
        vmax=+1,         # cap them at +1
        row_cluster=True,
        col_cluster=False,
        col_colors=[lut[mapping[s]] for s in data_ordered.columns],
        figsize=(12,10),        # widen figure for long labels
        yticklabels=True,
        xticklabels=True         # turn ON sample labels
    )

    # Rotate and align the x-axis tick labels (patient IDs)
    cg.ax_heatmap.set_xticklabels(
        cg.ax_heatmap.get_xmajorticklabels(),
        rotation=45,
        ha="right",
        fontsize=9
    )

    # Axis labels
    cg.ax_heatmap.set_ylabel("Top DE genes", fontsize=12)
    cg.ax_heatmap.set_xlabel("Patient ID", fontsize=12)

    #Separate the two treatment blocks with a white line
    split = sum(1 for s in data_ordered.columns if mapping[s]=="Untreated")
    cg.ax_heatmap.axvline(split, color="white", lw=2)

    # --- ADDED SECTION: Add labels directly above the color bars ---

    # Get the Axes object for the column color bar
    ax_colors = cg.ax_col_colors
    
    # Calculate the horizontal center for the "Untreated" block
    untreated_center = split / 2
    
    # Add the "Untreated" text
    ax_colors.text(untreated_center, 0.5, 'Untreated', 
                   ha='center', va='center', color='white', fontweight='bold')

    # Calculate the horizontal center for the "Treated" block
    total_cols = len(data_ordered.columns)
    treated_center = split + (total_cols - split) / 2
    
    # Add the "Treated" text
    ax_colors.text(treated_center, 0.5, 'Treated', 
                   ha='center', va='center', color='white', fontweight='bold')
    
    # --- END OF ADDED SECTION ---


    #Build a legend and place it *outside* to the right
    # This is now redundant, so we can comment it out to avoid clutter.
    # patches = [Patch(facecolor=color, label=grp) for grp, color in lut.items()]
    # cg.fig.legend(
    #     handles=patches,
    #     title="Treatment",
    #     loc="center right",
    #     bbox_to_anchor=(1.2, 0.5),  # x=1.2 pushes it outside the heatmap
    #     frameon=False
    # )

    plt.show() # Added plt.show() to ensure the plot displays

Plot the DGE heatmaps using the original settings.


In [ ]:
plot_DE(mac_clean, expr_z)
plot_DE(mac_clean, expr_z_up)
plot_DE(mac_clean, expr_z_down)


#### 1.6: Define the Macrophage Subtype with a strong therapy-associated gene signature. This signature shows to be antigen presenting in g:profiler. 


Score all macrophages based on the upregulated gene signature in the treated group.

In [ ]:
sc.tl.score_genes(
    mac_clean,
    gene_list  = up_genes,
    score_name = 'antigen_score'
)

Identify the top 10% of cells with the highest score *within each treatment group*

In [ ]:
mac_clean.obs['antigen_signature'] = False
for grp in mac_clean.obs['treatment'].unique():
    mask = mac_clean.obs['treatment'] == grp
    threshold = np.percentile(mac_clean.obs.loc[mask, 'antigen_score'], 90)
    mac_clean.obs.loc[mask, 'antigen_signature'] = (mac_clean.obs.loc[mask, 'antigen_score'] >= threshold)

In [ ]:
mac_clean.obs['antigen_signature'] = mac_clean.obs['antigen_signature'].astype(bool)
mac_clean.obs['antigen_signature'] = mac_clean.obs['antigen_signature'].astype('category')


### Part 2: Generalized Spatial Proximity Analysis

Here, we will calculate the spatial proximity of our newly defined macrophages with a strong antigen signature to all other cell types and test for significant differences between treatment groups.


#### 2.1: Prepare the Full Dataset for Spatial Analysis

In [ ]:
# Proximity calculations use only observation metadata.


Define an exponential decay model to calculte a proximity score of any 2 cell types per fov. 

In [ ]:
from pdac_macrophages.spatial import exponential_model



Create a clean copy of the full dataset

In [ ]:
adata2 = adata.copy()

Preprocess the full dataset (NaN cleaning and normalization)

In [ ]:
adata2.X = replace_nan_with_zero(adata2.X)
sc.pp.normalize_total(adata2, target_sum=1e4)
sc.pp.log1p(adata2)


Map the 'antigen_signature' status from mac_clean to the full dataset

In [ ]:
adata2.obs['antigen_signature'] = False
common = adata2.obs_names.intersection(mac_clean.obs_names)
adata2.obs.loc[common, 'antigen_signature'] = mac_clean.obs.loc[common, 'antigen_signature'].astype(bool)

Annotate treatment condition on the full dataset

In [ ]:
patient_id = adata2.obs["sample_id"].cat.categories
treated_patients = [id for id in patient_id if id.startswith("T")]
adata2.obs['treatment'] = np.where(adata2.obs['sample_id'].isin(treated_patients), 'Treated', 'Untreated')
adata2.obs['treatment'] = pd.Categorical(adata2.obs['treatment'], categories=['Untreated','Treated'])

Create a single cell type column that distinguishes "Antigen signature presenting Macrophages"

In [ ]:
adata2.obs['mac_type'] = np.where(
    (adata2.obs['annotation_majortypes']=='Macrophage') & (adata2.obs['antigen_signature'].astype(bool)),
    'Antigen signature presenting macrophages',
    adata2.obs['annotation_majortypes']
)
adata2.obs['mac_type'] = adata2.obs['mac_type'].astype('category')


#### 2.2: Calculate Proximity Scores for All Cell Type Pairs


Get the list of all cell types to analyse

In [ ]:
all_categories = adata2.obs['annotation_majortypes'].cat.categories.tolist()
exclude_types = ['Macrophage', 'uninformative_EVEN', 'CD4+ T', 'B', 
                 'Dendritic (activated)', 'Dendritic (conventional)', 'Dendritic (plasmacytoid)', 
                 'Endocrine', 'Natural killer', 'Neutrophil', 'Noise', 'Schwann/nerve', 'Smooth muscle', 'ductal']
target_cell_types = [ct for ct in all_categories if ct not in exclude_types]

In [ ]:
print("Will analyze interactions between Macrophages showing a strong antigen signature and the following cell types:")
print(target_cell_types)

In [ ]:
records = []
for cond in ['Treated','Untreated']:
    ad = adata2[adata2.obs['treatment']==cond, :]
    for cell_type in target_cell_types:
        for samp in ad.obs['sample_id'].cat.categories:
            sub_samp = ad[ad.obs['sample_id']==samp, :]
            d1, d2 = exponential_model(
                sub_samp,
                ct1     = 'Antigen signature presenting macrophages',
                ct2     = cell_type,
                ct1_obs = 'mac_type',
                ct2_obs = 'annotation_majortypes',
                radius  = 50
            )
            if len(d1) > 0: # Only append if there were antigen signature presenting macs to score
                records.append(pd.DataFrame({
                    'proximity': d1,
                    'interacting_cell': cell_type,
                    'Condition': cond,
                    'sample_id': samp
                }))


#### 2.3: Visualize and Statistically Test Proximity Differences


Concatenate results and create the main proximity DataFrame

In [ ]:
df_proximity = pd.concat(records, ignore_index=True)

Visualize proximity scores with a boxplot

In [ ]:
plt.figure(figsize=(20, 7))
ax = sns.boxplot(
    data=df_proximity,
    x='interacting_cell',
    y='proximity',
    hue='Condition',
    palette={"Treated": "blue", "Untreated": "red"}
)
plt.ylabel('Σ exp(-d / 50 µm) per antigen signature presenting MAC')
plt.title('Macrophage–Cell Spatial Proximity')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.legend(title='Condition', loc='upper right')
plt.tight_layout()
plt.show()


Compute per-patient median and mean proximity scores

In [ ]:
df_median_proximity = df_proximity.groupby(['Condition','sample_id','interacting_cell'])['proximity'].median().reset_index(name='median_proximity')
df_mean_proximity = df_proximity.groupby(['Condition','sample_id','interacting_cell'])['proximity'].mean().reset_index(name='mean_proximity')


Define outlier removal function

In [ ]:
from pdac_macrophages.statistics import outliers_iqr


In [ ]:
from pdac_macrophages.statistics import erase_outliers


**Original analysis choice retained:** remove IQR outliers within each cell type and condition. See the review before interpreting these tests.


In [ ]:
df_median_proximity = erase_outliers(df_median_proximity, 'median')
df_mean_proximity = erase_outliers(df_mean_proximity, 'mean')


Define function to test for normality (Shapiro) and variance equality (Levene)

In [ ]:
from scipy.stats import levene, shapiro

def parametric_assumptions_test(proximity_dataframe, mean_or_median): 
    """
    Checks if the data for each cell type meets the assumptions for parametric
    statistical tests (like a t-test).

    Specifically, it tests for:
    1. Normality: Are the data in each group (Treated/Untreated) normally distributed? (Shapiro-Wilk test)
    2. Homogeneity of variances: Do the two groups have similar variances? (Levene's test)

    Parameters
    ----------
    proximity_dataframe : pd.DataFrame
        The DataFrame containing per-patient mean or median proximity scores.
    mean_or_median : str
        A string, 'mean' or 'median', indicating which score column to use.

    Returns
    -------
    tuple of lists
        A tuple containing two lists:
        - The first list contains the names of cell types that *meet* both assumptions.
        - The second list contains the names of cell types that *fail* at least one assumption.
    """

    normal, not_normal, similar_var, not_similar_var = {}, {}, {}, {}

    for cell_type in proximity_dataframe['interacting_cell'].unique():
        grp = proximity_dataframe.query("interacting_cell == @cell_type")
        # Extract scores for the treated and untreated groups.
        treated = grp.query("Condition=='Treated'")[f'{mean_or_median}_proximity']
        untreated = grp.query("Condition=='Untreated'")[f'{mean_or_median}_proximity']
        if len(treated) < 3 or len(untreated) < 3: continue
        # null hypothesis to check if data follows a normal distribution, rejection (p less tham 0.05) means that data does not follow normal distribution
        stat_t, p_t = shapiro(treated); stat_u, p_u = shapiro(untreated) 
        if p_t <= 0.05 or p_u <= 0.05: not_normal[cell_type] = (p_t, p_u)
        else: normal[cell_type] = (p_t, p_u)
        # null hypothesis states equality of variances, p less than 0.05 rejects this hypothesis
        stat_v, p_v = levene(treated, untreated)
        if p_v <= 0.05: not_similar_var[cell_type] = p_v
        else: similar_var[cell_type] = p_v

    print(f"--- Assumption testing for {mean_or_median.upper()} proximity ---")

    if normal: print(f"Normal distribution in: {list(normal.keys())}")
    if not_normal: print(f"NOT normal in: {list(not_normal.keys())}")
    if similar_var: print(f"Similar variance in: {list(similar_var.keys())}")
    if not_similar_var: print(f"NOT similar variance in: {list(not_similar_var.keys())}")

    #Define lists satisfying parametric assumptions and not sutisfying parametric assumptions
    parametric_cells = [cell for cell in list(normal.keys()) if cell in list(similar_var.keys())]
    non_parametric_cells = sorted(set(not_normal) | set(not_similar_var))

    return parametric_cells, non_parametric_cells


Run assumption tests

In [ ]:
median_parametric_cells, median_non_parametric_cells = parametric_assumptions_test(df_median_proximity, 'median')

In [ ]:
print(median_non_parametric_cells)


Define functions for statistical testing to verify differences in proximities between treated and untreated patients. T-test for cell proximities that satisfy parametric assumptions and Mann-Whitney U for the rest. 

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu
def t_test(proximity_dataframe, median_or_mean, cell_list):
    """
    Performs an independent two-sample t-test to compare the mean proximity
    scores between treated and untreated groups.

    This function should only be used for cell types that have passed the
    parametric assumptions tests (normality and homogeneity of variances).

    Parameters
    ----------
    proximity_dataframe : pd.DataFrame
        DataFrame containing per-patient mean or median proximity scores.
    median_or_mean : str
        A string, 'mean' or 'median', indicating which score column to use.
    cell_list : list of str
        A list of the names of cell types to perform the test on.
    """
    if not cell_list: return
    print(f"\n--- Running T-test for {median_or_mean} proximity ---")
    for cell_type in cell_list:
        grp = proximity_dataframe.query("interacting_cell == @cell_type")
        treated = grp.query("Condition=='Treated'")[f'{median_or_mean}_proximity']
        untreated = grp.query("Condition=='Untreated'")[f'{median_or_mean}_proximity']
        stat, p = ttest_ind(treated, untreated, equal_var=True)
        print(f"  {cell_type:25s} {'(SIGNIFICANT)' if p <= 0.05 else ''} p = {p:.3g}")

In [ ]:
def mannwhitneyutest(proximmity_databse, mean_or_median, cell_list):
    """
    Performs a two-sided Mann-Whitney U-test (a non-parametric equivalent
    of the t-test) to compare proximity score distributions between groups.

    This function should be used for cell types that failed the parametric
    assumptions tests (i.e., data is not normal or variances are not equal).
    """
    if not cell_list: return
    print(f"\n--- Running Mann-Whitney U-test for {mean_or_median} proximity ---")
    for cell_type in cell_list:
        grp = proximmity_databse.query("interacting_cell == @cell_type")
        treated = grp.query("Condition=='Treated' & interacting_cell==@cell_type")[f'{mean_or_median}_proximity']
        untreated = grp.query("Condition=='Untreated' & interacting_cell==@cell_type")[f'{mean_or_median}_proximity']
        if len(treated) == 0 or len(untreated) == 0: continue
        stat, p = mannwhitneyu(treated, untreated, alternative='two-sided')
        print(f"  {cell_type:25s} {'(SIGNIFICANT)' if p <= 0.05 else ''} p = {p:.3g}")

Run the appropriate statistical test based on assumption checks

In [ ]:
t_test(df_median_proximity, 'median', cell_list=median_parametric_cells)
mannwhitneyutest(df_median_proximity, 'median', cell_list=median_non_parametric_cells)

In [ ]:
t_test(df_mean_proximity, 'mean', cell_list=median_parametric_cells)
mannwhitneyutest(df_mean_proximity, 'mean', cell_list=median_non_parametric_cells)



### Part 3: Generalized Ligand-Receptor Analysis Pipeline<br>

This final section contains all the functions needed for the L-R analysis and a master loop to execute the entire pipeline for all `target_cell_types`.


#### 3.1: Define All Helper Functions for the L-R Pipeline

In [ ]:
import scotia
import liana.method as lm
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches


Set the main output folder path once

In [ ]:
ligand_receptor_folder = OUTPUT_ROOT


Define function to find potential ligand receptor pairs, using databases supported by liana(e.g. connectome, cellphonedb) 

In [ ]:
def export_LR_pairs_full(adata, database, cell_pairs, out_root=ligand_receptor_folder):
    """"
    adata : AnnData
        The full AnnData object containing all cells. Must have an .obs column
        named 'mac_type' for cell type annotations.
    database : str
        The name of the LIANA method to use (e.g., 'connectome', 'cellphonedb').
    cell_pairs : tuple of tuples
        A tuple containing one or more tuples, each representing a
        (ligand_cell_type, receptor_cell_type) pair to analyze.
    out_root : Path, optional
        The root directory for saving the output files.
    """
    adata.obs['cell_type'] = adata.obs['mac_type']
    out_base = out_root / f'L-R {database} full'
    out_base.mkdir(parents=True, exist_ok=True)
    method = getattr(lm, database)
    key = f"{database}_full"
    method(adata, groupby='cell_type', resource_name='consensus', expr_prop=0.1, n_perms=1000, use_raw=False, key_added=key, verbose=False)
    lr_df = adata.uns[key] #dataframe of potential l-r pairs. 
    
    for ligand, receptor in cell_pairs:
        ligand2receptor = lr_df.query(f"(source == '{ligand}' and target == '{receptor}') or (source == '{receptor}' and target == '{ligand}')")
        out_csv = out_base / f"liana_{ligand}_to_{receptor}_full.csv"
        ligand2receptor.to_csv(out_csv, index=False)

In [ ]:
from pdac_macrophages.ligand_receptor import combine_top_hits


In [ ]:
from pdac_macrophages.ligand_receptor import subset_lr_data


In [ ]:
def run_scotia_loopable(adata_sub, lr_list, ligand, receptor, out_root=ligand_receptor_folder):
    for cond in ['Treated','Untreated']:
        ad_cond = adata_sub[adata_sub.obs['treatment'] == cond].copy()
        outdir  = out_root / f'scotia_{ligand}_{receptor}_{cond.lower()}_full'
        outdir.mkdir(exist_ok=True, parents=True)
        scotia.run_scotia.lr_score(ad_cond, lr_list, sample_col='sample_id', fov_col='fov', celltype_col='mac_type', output_path=str(outdir))

In [ ]:
from pdac_macrophages.ligand_receptor import load_scotia_results


In [ ]:
from pdac_macrophages.ligand_receptor import summarize_lr


In [ ]:
from pdac_macrophages.ligand_receptor import add_between_group_pvals


In [ ]:
def plot_lr_fold_change(comp_df, title_suffix=""):
    """
    Generates a bar plot to visualize significant changes in ligand-receptor
    interactions between two treatment groups.

    Parameters
    ----------
    comp_df : pd.DataFrame
        A DataFrame containing statistical comparison results. Must include columns:
        'ligand', 'receptor', 'log2FC_T_over_U', and 'qval'.
    title_suffix : str, optional
        A string to append to the plot title, often used to identify the
        specific cell-cell interaction being plotted (e.g., "Macrophage <-> CAF").
    """
    if comp_df.empty:
        print(f"No comparable L-R pairs for {title_suffix}. Skipping plot.")
        return
    # Filter for statistically significant interactionse
    sig_df = comp_df[comp_df['qval'] < 0.05].copy()
    if sig_df.empty:
        print(f"No significant interactions (qval < 0.05) to plot for {title_suffix}.")
        return

    sig_df['lr_pair'] = sig_df['ligand'] + '–' + sig_df['receptor']
    sig_df = sig_df.sort_values('log2FC_T_over_U', ascending=False).reset_index(drop=True) # Sort DataFrame by log2 fold change.
    sig_df['color'] = sig_df['log2FC_T_over_U'].apply(lambda fc: 'blue' if fc > 0 else 'red')  # Positive fold change (enriched in treated) will be blue, Negative fold change (enriched in untreated) will be red.
    fig, ax = plt.subplots(figsize=(max(10, len(sig_df) * 0.4), 7))
    ax.bar(sig_df.index, sig_df['log2FC_T_over_U'], color=sig_df['color'], width=0.8)

    for i, row in sig_df.iterrows():
        y_pos = row['log2FC_T_over_U'] + (0.03 if row['log2FC_T_over_U'] > 0 else -0.05)
        ax.text(i, y_pos, '●', ha='center', va='bottom' if row['log2FC_T_over_U'] > 0 else 'top', fontsize=12)

    ax.set_xticks(sig_df.index); ax.set_xticklabels(sig_df['lr_pair'], rotation=90, ha='center', fontsize=10)
    ax.set_ylabel('log₂(LR interaction fold change)', fontsize=14)
    ax.set_title(f'Significant L-R Interaction Changes Between Treatment Groups: {title_suffix}', fontsize=16)
    ax.axhline(0, color='black', linewidth=0.8); ax.spines[['top', 'right']].set_visible(False)
    ymin, ymax = ax.get_ylim(); ax.set_ylim(ymin - 0.1, ymax + 0.1)
    blue = mpatches.Patch(color='blue', label='Enriched in treated')
    red = mpatches.Patch(color='red', label='Enriched in untreated')
    color_legend = ax.legend(handles=[red, blue], loc='upper right', frameon=False)
    ax.add_artist(color_legend)
    symbol_legend = [Line2D([0], [0], marker='o', color='w', label='Mann-Whitney U-test (adj. P < 0.05)', markerfacecolor='k', markersize=8)]
    ax.legend(handles=symbol_legend, loc='upper right', bbox_to_anchor=(1, 0.9), frameon=False)
    plt.tight_layout(); plt.show()


#### 3.2: Prepare Data for SCOTIA 


Add spatial coordinates in microns to the main AnnData object

In [ ]:
coords_px = adata2.obs[['CenterX_local_px','CenterY_local_px']].to_numpy()
um_per_px = 0.18 # This ratio is specific to the instrument/dataset
coords_um = coords_px * um_per_px
adata2.obsm['spatial'] = coords_um
# Historical SCOTIA compatibility workaround; see docs/REVIEW.md.
np.Inf = np.inf


#### 3.3: Master Loop to Run Full L-R Pipeline for All Cell Types

In [ ]:
if not RUN_LR_ANALYSIS:
    raise RuntimeError("LIANA/SCOTIA execution is disabled. Set RUN_LR_ANALYSIS = True to opt in.")

for receptor_cell in target_cell_types:
    print(f"\n\n{'='*60}\n      RUNNING L-R PIPELINE FOR: Antigen signature presenting Macrophages <-> {receptor_cell}\n{'='*60}\n")
    ligand_cell = 'Antigen signature presenting macrophages'
    cell_pair_tuple = [(ligand_cell, receptor_cell)]

    # STEP 1: RUN LIANA
    print("--- STEP 1: Running LIANA ---")
    try:
        export_LR_pairs_full(adata2, 'connectome', cell_pair_tuple)
        export_LR_pairs_full(adata2, 'cellphonedb', cell_pair_tuple)
    except Exception as e:
        print(f"  ERROR during LIANA: {e}. Skipping {receptor_cell}.")
        continue

    # STEP 2: COMBINE L-R LISTS
    print("\n--- STEP 2: Combining L-R hits for SCOTIA ---")
    try:
        lr_list = combine_top_hits(ligand_cell, receptor_cell, out_root=ligand_receptor_folder)
        if lr_list.empty: print(f"  No L-R pairs found for {receptor_cell}. Skipping."); continue
    except FileNotFoundError:
        print(f"  LIANA output not found for {receptor_cell}. Skipping."); continue

    # STEP 3: RUN SCOTIA
    print("\n--- STEP 3: Subsetting data and running SCOTIA ---")
    adata_sub = subset_lr_data(adata2, ligand_cell, receptor_cell)
    if receptor_cell not in adata_sub.obs['mac_type'].unique():
        print(f"  No cells of type '{receptor_cell}' found. Skipping."); continue
    run_scotia_loopable(adata_sub, lr_list, ligand_cell, receptor_cell)

    # STEP 4: LOAD & SUMMARIZE SCOTIA RESULTS
    print("\n--- STEP 4: Loading and summarizing SCOTIA results ---")
    try:
        treated_df = load_scotia_results(ligand_receptor_folder / f"scotia_{ligand_cell}_{receptor_cell}_treated_full/ot")
        untreated_df = load_scotia_results(ligand_receptor_folder / f"scotia_{ligand_cell}_{receptor_cell}_untreated_full/ot")
        if treated_df.empty or untreated_df.empty:
            print(f"  SCOTIA output empty for one or both conditions. Skipping stats."); continue
        treated_summary = summarize_lr(treated_df)
        untreated_summary = summarize_lr(untreated_df)
    except Exception as e:
        print(f"  ERROR loading SCOTIA results for {receptor_cell}: {e}. Skipping."); continue

    # STEP 5: CALCULATE P-VALUES & Q-VALUES
    print("\n--- STEP 5: Calculating statistical significance ---")
    csv_path = ligand_receptor_folder / "scotia_l-r_pairs_results"
    csv_path.mkdir(exist_ok=True, parents=True)
    treated_csv = csv_path / f"summary_treated_{ligand_cell}_to_{receptor_cell}.csv"
    untreated_csv = csv_path / f"summary_untreated_{ligand_cell}_to_{receptor_cell}.csv"
    comp_df = add_between_group_pvals(treated_df, untreated_df, treated_summary, untreated_summary, treated_csv, untreated_csv)

In [ ]:
# loop through results and call the plotting function.
def plot_all_lr_results(target_cell_types, ligand_cell='Antigen signature presenting macrophages', out_root=ligand_receptor_folder):
    """
    Loads the final summary CSV files generated by the pipeline and calls the
    plotting function for each one.
    """
    print(f"\n{'='*60}\n      GENERATING ALL FINAL PLOTS\n{'='*60}\n")
    
    for receptor_cell in target_cell_types:
        print(f"--- Checking results for: {ligand_cell} <-> {receptor_cell} ---")
        
        # Construct the path to the result file. We only need to load one (e.g., the treated one)
        # as the `add_between_group_pvals` function saves the full comparison df to both.
        csv_path = out_root / "scotia_l-r_pairs_results" / f"summary_treated_{ligand_cell}_to_{receptor_cell}.csv"

        try:
            # Load the final DataFrame that contains the p-values and q-values
            comp_df = pd.read_csv(csv_path)
            
            # The comp_df might have NaNs in the stats columns for pairs that were not
            # comparable. We need to drop these before plotting.
            comp_df.dropna(subset=['qval'], inplace=True)

            # Call the plotting function
            plot_lr_fold_change(
                comp_df,
                title_suffix=f"Macrophages <-> {receptor_cell}"
            )

        except FileNotFoundError:
            print(f"  Result file not found for {receptor_cell}. Skipping plot.")
            continue
        except Exception as e:
            print(f"  An error occurred while plotting for {receptor_cell}: {e}")
            continue

In [ ]:
# --- FINAL CALL TO GENERATE PLOTS ---
# This will now run the plotting sequence.
plot_all_lr_results(target_cell_types)